In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

spark

25/09/09 12:31:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
%%sql

CREATE DATABASE IF NOT EXISTS poc

++
||
++
++

In [5]:
%%sql

CREATE TABLE IF NOT EXISTS iceberg.ip.nyc_taxi_zone_ice (
    location_id VARCHAR(20),
    borough VARCHAR(50),
    zone VARCHAR(20),
    service_zone VARCHAR(50)
)
USING iceberg;

++
||
++
++

In [6]:
%%sql
DESCRIBE EXTENDED iceberg.ip.nyc_taxi_zone_ice;  

col_name,data_type,comment
location_id,string,None
borough,string,None
zone,string,None
service_zone,string,None
,,
# Metadata Columns,,
_spec_id,int,
_partition,struct<>,
_file,string,
_pos,bigint,


In [7]:
%%sql
select * from hive.hp.nyc_taxi_zone_hive;

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `hive`.`hp`.`nyc_taxi_zone_hive` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 14;
'Project [*]
+- 'UnresolvedRelation [hive, hp, nyc_taxi_zone_hive], [], false


In [3]:
%%sql

CREATE TABLE IF NOT EXISTS poc.events (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg;

++
||
++
++

In [5]:
# Create DataFrame
data = [(1, "James", 30, "M"), (2, "Ann", 40, "F")]
columns = ["id", "name", "age", "gender"]
sampleDF = spark.sparkContext.parallelize(data).toDF(columns)

In [6]:
# Create a Hive Internal table
sampleDF.write.mode("overwrite").saveAsTable("emp.employee")

In [7]:
%%sql

select * from emp.employee;

id,name,age,gender
1,James,30,M
2,Ann,40,F


In [10]:
%%sql
DESCRIBE EXTENDED emp.employee;   

col_name,data_type,comment
id,bigint,None
name,string,None
age,bigint,None
gender,string,None
,,
# Metadata Columns,,
_spec_id,int,
_partition,struct<>,
_file,string,
_pos,bigint,


In [11]:
%%sql

CREATE DATABASE IF NOT EXISTS nyc

++
||
++
++

In [12]:
df = spark.read.parquet("/home/iceberg/data/yellow_tripdata_2025-01.parquet")
df.write.saveAsTable("nyc.taxis")

In [13]:
%%sql

DESCRIBE EXTENDED nyc.taxis

col_name,data_type,comment
VendorID,int,None
tpep_pickup_datetime,timestamp_ntz,None
tpep_dropoff_datetime,timestamp_ntz,None
passenger_count,bigint,None
trip_distance,double,None
RatecodeID,bigint,None
store_and_fwd_flag,string,None
PULocationID,int,None
DOLocationID,int,None
payment_type,bigint,None


In [14]:
%%sql

SELECT COUNT(*) as cnt
FROM nyc.taxis

cnt
3475226
